# lib_eval_add — lib_eval 보강 함수 (작성·검증: eval 담당 2026-11-26)

`lib_eval.ipynb`를 먼저 `%run -i`한 뒤 이 노트북을 `%run -i ./lib_eval_add.ipynb`로 불러 씁니다. `lib_eval`의 함수는 고치지 않고 아래 4가지를 추가합니다.

- `paired_metric_ci`, `pom_koklev`: 6-A. 같은 재표본으로 두 모형을 평가한 paired bootstrap, Koklev(2025) III.G 부호(비용이 크면 항상 +)
- `sr_point_mode`, `paired_bootstrap_sr_mode`: SR 정의 두 가지(with_treasury / approved_only)
- `swap_set`: 6-B 같은 승인율에서 ref만, new만, 둘 다 승인·거절한 집단의 규모·부도율·초과수익률
- `delta_sr_curve`: 승인율별 ΔSR

가짜 데이터 점검 10개 통과(같은 p̂ → PoM·ΔSR 0, ③이 나쁘면 PoM 모두 +, 승인율 1.0에서 두 SR 모드 일치 등). 66.6만 건 기준 approved_only 4,999회 약 2분, 6-A paired CI 1,000회 약 11분.

In [ ]:
# lib_eval 보강 (eval 담당, 2026-11-26). lib_eval.ipynb를 %run -i 한 다음 이 칸을 실행한다. 기존 함수는 고치지 않는다.
# 포함: pom_koklev, paired_metric_ci(6-A), sr_point_mode / paired_bootstrap_sr_mode (SR 정의 2가지), swap_set, delta_sr_curve
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

def pom_koklev(metric, ref, new):
    """Koklev(2025) III.G 정의. 비용이 클수록 항상 +. auc·ks: (ref−new)/ref, brier·logloss: (new−ref)/ref"""
    return (ref - new) / ref if metric in ("auc", "ks") else (new - ref) / ref

def _metric(name, y, p):
    if name == "auc": return roc_auc_score(y, p)
    if name == "ks": return stats.ks_2samp(p[y == 1], p[y == 0]).statistic
    if name == "brier": return brier_score_loss(y, p)
    if name == "logloss": return log_loss(y, np.clip(p, 1e-15, 1 - 1e-15))
    raise KeyError(name)

def paired_metric_ci(y, p_ref, p_new, n_boot=1000, seed=42, metrics=("auc", "ks", "brier", "logloss")):
    """6-A: 같은 재표본으로 두 모형을 평가한 paired bootstrap. PoM(Koklev 부호)과 percentile 95% CI."""
    y = np.asarray(y); n = len(y); rng = np.random.default_rng(seed); rows = []
    idxs = [rng.integers(0, n, n) for _ in range(n_boot)]
    for m in metrics:
        pt_ref, pt_new = _metric(m, y, p_ref), _metric(m, y, p_new)
        bs = np.empty(n_boot)
        for i, ix in enumerate(idxs):
            if y[ix].min() == y[ix].max(): bs[i] = np.nan; continue
            bs[i] = pom_koklev(m, _metric(m, y[ix], p_ref[ix]), _metric(m, y[ix], p_new[ix]))
        bs = bs[~np.isnan(bs)]
        rows.append(dict(metric=m, ref=pt_ref, new=pt_new, diff=pt_new - pt_ref,
                         pom=pom_koklev(m, pt_ref, pt_new), pom_lo=np.quantile(bs, .025), pom_hi=np.quantile(bs, .975)))
    return pd.DataFrame(rows)

def sr_point_mode(p, e, a, mode="with_treasury", w=None):
    """SR_MODE 두 가지. with_treasury = 기존 sr_point(거절=0, 전체 n). approved_only = 승인된 k건만."""
    if mode == "with_treasury": return sr_point(p, e, a, w)
    m = approve_mask(p, a); r = e[m]
    if w is None: return r.mean() / r.std(ddof=1)
    ww = np.asarray(w, float)[m]; ww = ww / ww.sum(); mu = (ww * r).sum()
    return mu / np.sqrt((ww * (r - mu) ** 2).sum() * len(r) / (len(r) - 1))

def _sr_counts_approved(order, cnt, e_sorted, k):
    c = cnt[order]; before = np.cumsum(c) - c; take = np.clip(k - before, 0, c)
    s1 = (take * e_sorted).sum(); s2 = (take * e_sorted * e_sorted).sum(); mean = s1 / k
    return mean / np.sqrt((s2 - k * mean * mean) / (k - 1))

def paired_bootstrap_sr_mode(P, e, a, n_boot, seed=SEED, mode="with_treasury"):
    """paired_bootstrap_sr의 SR 정의 선택판. with_treasury는 기존 함수를 그대로 부름."""
    if mode == "with_treasury": return paired_bootstrap_sr(P, e, a, n_boot, seed)
    n = len(e); k = n_approve(n, a); rng = np.random.default_rng(seed)
    orders = {m: np.argsort(p, kind="stable") for m, p in P.items()}; es = {m: e[o] for m, o in orders.items()}
    out = {m: np.empty(n_boot) for m in P}
    for i in range(n_boot):
        cnt = np.bincount(rng.integers(0, n, n), minlength=n)
        for m, o in orders.items(): out[m][i] = _sr_counts_approved(o, cnt, es[m], k)
    return pd.DataFrame(out)

def swap_set(p_ref, p_new, y, e, a):
    """6-B swap set: 같은 승인율에서 ref만 승인 / new만 승인 / 둘 다 승인 / 둘 다 거절 집단의 규모·부도율·초과수익률."""
    A, B = approve_mask(p_ref, a), approve_mask(p_new, a)
    groups = {"둘 다 승인": A & B, "ref만 승인": A & ~B, "new만 승인": ~A & B, "둘 다 거절": ~A & ~B}
    rows = [dict(집단=g, n=int(m.sum()), 비율=m.mean(), 부도율=float(y[m].mean()) if m.any() else np.nan,
                 평균_초과수익률=float(e[m].mean()) if m.any() else np.nan) for g, m in groups.items()]
    return pd.DataFrame(rows)

def delta_sr_curve(p_ref, p_new, e, rates, mode="with_treasury"):
    """승인율별 ΔSR = SR(ref) − SR(new)."""
    return pd.DataFrame({"rate": list(rates), "dSR": [sr_point_mode(p_ref, e, r, mode) - sr_point_mode(p_new, e, r, mode) for r in rates]}).set_index("rate")["dSR"]
